In [26]:
%pip -q install openai-whisper jiwer pandas
# Make sure ffmpeg is installed locally (e.g. `brew install ffmpeg` on macOS)
import subprocess, shutil
if not shutil.which("ffmpeg"):
    print("WARNING: ffmpeg not found. Install it: brew install ffmpeg (macOS) or apt install ffmpeg (Linux)")
else:
    print("ffmpeg found.")
print("Dependencies ready.")

Note: you may need to restart the kernel to use updated packages.
ffmpeg found.
Dependencies ready.


In [27]:
import os
import re
import time
import whisper
import pandas as pd
from IPython.display import display

AUDIO_DIR = "./audio/selected_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

_model_cache = {}

def get_model(name="base"):
    if name not in _model_cache:
        print(f"Loading Whisper model '{name}'...")
        _model_cache[name] = whisper.load_model(name)
    return _model_cache[name]

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    return text

def read_csv_auto(path):
    for enc in ["utf-8", "utf-8-sig", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="latin-1")

print("Imports and helpers ready.")

Imports and helpers ready.


In [28]:
INTENT_CONFIG = {
    "decision": {
        "tower.decision.right": [["right"], ["2"], ["two"], ["second"]],
        "tower.decision.left": [["left"], ["1"], ["number", "one"], ["first"]],
        "tower.decision.same": [["both"], ["together"], ["same", "time"], ["one", "and", "two"]]
    },
    "reason": {
        "tower.reason.tall": [["tall"], ["taller"], ["big"], ["bigger"], ["high"], ["higher"]],
        "tower.reason.weight": [["weight"], ["weighs"], ["top"], ["heavy"], ["heavier"]],
        "tower.reason.thinnerbase": [["thin"], ["thinner"], ["small"], ["smaller"], ["base"], ["bottom"]],
        "tower.reason.notsymmetrical": [["symmetry"], ["symmetric"], ["symmetrical"], ["both", "sides"], ["not", "the", "same"], ["asymmetrical"], ["asymmetric"], ["not", "same"]]
    }
}

def determine_intent(transcript, state):
    """Implementation of the 3-step intent matching logic."""
    if not transcript or str(transcript).strip() == "":
        return "default.noaudio"

    # Step 1: Clean text
    clean = normalize_text(transcript)
    words = clean.split()

    # Step 2: Get intents for current state
    state_intents = INTENT_CONFIG.get(state, {})

    # Step 3: Keyword group matching
    for intent_name, groups in state_intents.items():
        for group in groups:
            # Match if EVERY keyword in group is found as a whole word
            if all(any(re.fullmatch(re.escape(k), w) for w in words) for k in group):
                return intent_name

    return "default.unknown" # Or noaudio if you prefer

In [29]:
INPUT_CSV = "./reason_prediction_audio_log.csv"
WHISPER_MODEL_BATCH = "base"  # options: "tiny", "base", "small", "medium", "large"
BATCH_STATE = "default"       # options: "decision", "reason", "default"

KEYWORDS = {
    "reason": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric"
    ],
    "decision": [
        "right", "two", "left", "one", "both", "together", "same time",
    ],
    "default": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric",
        "right", "two", "left", "one", "both", "together", "same time",
    ],
}

def run_batch_pipeline(csv_path, model_name, state):
    if not os.path.exists(csv_path):
        print(f"Error: CSV not found at {csv_path}")
        print(f"Place your CSV file at: {os.path.abspath(csv_path)}")
        return

    df = read_csv_auto(csv_path)

    transcript_col = "re-run transcription"
    intent_col = f"re-run intent [{state}]"
    timing_col = "re-run processing time [ms]"

    for col in [transcript_col, intent_col, timing_col]:
        if col not in df.columns:
            df[col] = ""

    model = get_model(model_name)
    processed_count = 0
    skipped_count = 0
    error_count = 0

    keywords = KEYWORDS.get(state, KEYWORDS["default"])
    batch_prompt = "Try to focus specifically on these words as you listen to the audio: " + ", ".join(keywords)
    print(f"INITIAL PROMPT [{state}]: {batch_prompt}")

    print(f"Starting batch process for state: {state} on {len(df)} rows...\n")

    for idx, row in df.iterrows():
        if str(row.get(intent_col, '')).strip() not in ["", "nan"]:
            continue

        file_name = str(row['fileName']).strip()
        wav_path = os.path.join(AUDIO_DIR, file_name)

        if not os.path.exists(wav_path):
            skipped_count += 1
            continue

        existing_text = str(row.get(transcript_col, '')).strip()
        if existing_text in ["", "nan"]:
            try:
                start_time = time.time()
                result = model.transcribe(wav_path, initial_prompt=batch_prompt, language="en")
                elapsed_ms = (time.time() - start_time) * 1000

                new_text = result["text"].strip()
                df.at[idx, transcript_col] = new_text
                df.at[idx, timing_col] = round(elapsed_ms, 2)
            except Exception as e:
                print(f"Row {idx} ERROR: {file_name} — {e}")
                df.at[idx, transcript_col] = "ERROR"
                error_count += 1
                continue
        else:
            new_text = existing_text

        new_intent = determine_intent(new_text, state)

        df.at[idx, intent_col] = new_intent
        processed_count += 1
        print(f"Row {idx} processed: {file_name}")

    df.to_csv(csv_path, index=False)
    print(f"\nBatch Finished!")
    print(f"Processed: {processed_count} | Skipped: {skipped_count} | Errors: {error_count}")
    display(df.head(10))

run_batch_pipeline(INPUT_CSV, WHISPER_MODEL_BATCH, BATCH_STATE)

Loading Whisper model 'base'...
INITIAL PROMPT [default]: Try to focus specifically on these words as you listen to the audio: tall, taller, big, high, higher, weight, weighs, top, heavy, heavier, thin, thinner, small, smaller, symmetric, symmetrical, both sides, asymmetrical, asymmetric, right, two, left, one, both, together, same time
Starting batch process for state: default on 124 rows...



/usr/local/anaconda3/lib/python3.8/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Row 98 processed: 2026-6-14--15-27--18.wav

Batch Finished!
Processed: 1 | Skipped: 123 | Errors: 0


,date,time,duration_ms,filePath,fileName,fileSize_bytes,format,sampleRate,transcription,confidence,filePath.1,Process Time,Intent,Manual Transcription,Speaker,re-run transcription,re-run intent [default],re-run processing time [ms]
0,6/14/2026,10:14:10,6094,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-14--3.wav,153646,WAV,16000,cut up a cut up in our base.,0.4776,C:\Users\noril\Downloads\processing-3.5.4\data...,1.385048,tower.reason.thinnerbase,Because it has a thinner base,Norilla,,,
1,6/14/2026,10:14:24,6009,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-14--4.wav,153646,WAV,16000,with the head of inner base.,0.5231,C:\Users\noril\Downloads\processing-3.5.4\data...,1.317288,tower.reason.thinnerbase,Because it had a thinner base,Norilla,,,
2,6/14/2026,10:14:37,6060,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-14--5.wav,153646,WAV,16000,OVER 5 SECONDS,0.0000,C:\Users\noril\Downloads\processing-3.5.4\data...,0.000000,default.noaudio,NO SPOKEN WORD,NaN,,,
3,6/14/2026,10:18:07,6039,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-18--2.wav,163886,WAV,16000,"we'll come back and have more weight, just one...",0.4027,C:\Users\noril\Downloads\processing-3.5.4\data...,1.601501,tower.reason.weight,Because it has more weight on top than bottom,Norilla,,,
4,6/14/2026,10:18:45,6004,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-18--4.wav,163886,WAV,16000,because it has a thinner base.,0.5083,C:\Users\noril\Downloads\processing-3.5.4\data...,1.447961,tower.reason.thinnerbase,Because it has a thinner base,Norilla,,,
5,6/14/2026,10:22:44,6057,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-22--6.wav,163886,WAV,16000,in that case we have to make it a little bit s...,0.4194,C:\Users\noril\Downloads\processing-3.5.4\data...,1.845466,tower.reason.tall,Yeah. In that case we have to make it cause i...,Norilla,,,
6,6/14/2026,10:23:55,6027,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-23--8.wav,163886,WAV,16000,it's easy because that's how we're once taller...,0.4802,C:\Users\noril\Downloads\processing-3.5.4\data...,1.646755,tower.reason.tall,Because that tower is taller than the other tower,Norilla,,,
7,6/14/2026,10:24:33,6091,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-24--10.wav,163886,WAV,16000,in color that has a smaller base.,0.4298,C:\Users\noril\Downloads\processing-3.5.4\data...,1.355473,tower.reason.thinnerbase,Because it has a smaller base,Norilla,,,
8,6/14/2026,10:25:15,6100,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-25--12.wav,163886,WAV,16000,and not the same on both sides.,0.4395,C:\Users\noril\Downloads\processing-3.5.4\data...,1.760956,tower.reason.notsymmetrical,Um It's not the same on both sides,Norilla,,,
9,6/14/2026,10:26:55,6042,C:\Users\noril\Downloads\processing-3.5.4\data...,2026-6-14--10-26--15.wav,153646,WAV,16000,"Hello, the bottom is the body straight.",0.3684,C:\Users\noril\Downloads\processing-3.5.4\data...,2.693729,NaN,no the bottom is,User,,,


In [30]:
def run_intent_classification(csv_path):
    if not os.path.exists(csv_path):
        print(f"Error: CSV not found at {csv_path}")
        return

    df = read_csv_auto(csv_path)

    transcript_col = "re-run transcription"
    decision_col = "re-run intent [decision]"
    reason_col = "re-run intent [reason]"

    if transcript_col not in df.columns:
        print(f"No '{transcript_col}' column found.")
        return

    for col in [decision_col, reason_col]:
        if col not in df.columns:
            df[col] = ""

    classified = 0
    for idx, row in df.iterrows():
        transcript = str(row.get(transcript_col, '')).strip()
        if transcript in ["", "nan", "ERROR"]:
            continue

        df.at[idx, decision_col] = determine_intent(transcript, "decision")
        df.at[idx, reason_col] = determine_intent(transcript, "reason")
        classified += 1

    df.to_csv(csv_path, index=False)
    print(f"Classified {classified} rows for both decision and reason intents.")
    display(df[[transcript_col, decision_col, reason_col]].head(20))

run_intent_classification(INPUT_CSV)

Classified 1 rows for both decision and reason intents.


,re-run transcription,re-run intent [decision],re-run intent [reason]
0,NaN,,
1,NaN,,
2,NaN,,
3,NaN,,
4,NaN,,
5,NaN,,
6,NaN,,
7,NaN,,
8,NaN,,
9,NaN,,
